# HW06 — Data Preprocessing (Fill/Drop/Normalize)


In [ ]:
import os, sys, glob
from pathlib import Path
import pandas as pd

sys.path.append("src")
from config import get_data_dir_raw, get_data_dir_processed
from cleaning import fill_missing_median, drop_missing, normalize_data

RAW = get_data_dir_raw(); PRO = get_data_dir_processed()
RAW.mkdir(parents=True, exist_ok=True); PRO.mkdir(parents=True, exist_ok=True)

# Load a dataset from data/raw (or synthesize one)
candidates = sorted(glob.glob(str(RAW / "*.csv")))
if candidates:
    raw_path = candidates[-1]
    df_raw = pd.read_csv(raw_path)
else:
    # synthesize a tiny example
    raw_path = RAW / "toy_raw.csv"
    df_raw = pd.DataFrame({"x":[1,2,None,4], "y":[10.0,None,30.0,40.0], "cat":["a","b","b","a"]})
    df_raw.to_csv(raw_path, index=False)

print("Loaded:", raw_path)
display(df_raw.head())

# Apply cleaning functions
df_filled = fill_missing_median(df_raw)
df_dropped = drop_missing(df_filled, how="any")  # drop rows that still have NA
df_norm = normalize_data(df_dropped)

# Save cleaned
out_path = PRO / "cleaned_dataset.csv"
df_norm.to_csv(out_path, index=False)
print("Saved cleaned dataset:", out_path)

# Compare original vs cleaned
print("Original shape:", df_raw.shape, "-> Cleaned shape:", df_norm.shape)
display(df_raw.isna().sum().to_frame("NA_raw"))
display(df_norm.describe(include='number').T)


**Assumptions & tradeoffs**:
- Median fill applied to numeric columns only.
- Rows with remaining NAs are dropped (`how='any'`).
- Normalization uses z-score; zero-variance columns are mean-centered.
